# Projeto Data Lake - Arquitetura Lakehouse

## Diagrama Estrela (Star Schema)

```
                    ┌──────────────┐
                    │  dim_date    │
                    │──────────────│
                    │ date_key     │
                    │ date         │
                    │ year         │
                    │ month        │
                    │ day          │
                    │ day_of_week  │
                    │ quarter      │
                    │ is_weekend   │
                    └──────┬───────┘
                           │
┌──────────────┐   ┌───────┴────────┐   ┌──────────────┐
│ dim_stations │   │  fact_trips    │   │ dim_riders   │
│──────────────│   │────────────────│   │──────────────│
│ station_id   │◄──│ trip_id        │──►│ rider_id     │
│ station_name │   │ rideable_type  │   │ first_name   │
│ latitude     │   │ started_at     │   │ last_name    │
│ longitude    │   │ ended_at       │   │ address      │
└──────────────┘   │ start_station  │   │ birthday     │
                   │ end_station    │   │ start_date   │
                   │ rider_id       │   │ end_date     │
                   │ duration_min   │   │ is_member    │
                   └────────────────┘   └──────┬───────┘
                                               │
                   ┌────────────────┐          │
                   │ fact_payments  │          │
                   │────────────────│          │
                   │ payment_id     │──────────┘
                   │ payment_date   │
                   │ amount         │
                   │ rider_id       │
                   │ date_key       │
                   └────────────────┘
```

## Arquitetura de Camadas

| Camada | Descrição | Local |
|--------|-----------|-------|
| **Bronze** | Dados brutos (raw) importados dos CSVs | `/delta/bronze/` |
| **Silver** | Dados limpos, tipados e referenciados | `/delta/silver/` |
| **Gold** | Star schema otimizado para análise | `/delta/gold/` |

## Fontes de Dados
- `riders.csv` (75.000 registros) - Ciclistas/membros
- `payments.csv` (1.946.607 registros) - Pagamentos
- `stations.csv` - Estações de bicicleta
- `trips.csv` - Viagens realizadas

In [0]:
# ============================================
# CAMADA BRONZE - Dados brutos em Delta Lake
# ============================================

# Riders
bronze_riders = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "false") \
    .load("dbfs:/FileStore/tables/demo/Out/riders.csv")

bronze_riders.write.format("delta").mode("overwrite").save("/delta/bronze/riders")

# Payments
bronze_payments = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "false") \
    .load("dbfs:/FileStore/tables/demo/Out/payments.csv")

bronze_payments.write.format("delta").mode("overwrite").save("/delta/bronze/payments")

# Stations
bronze_stations = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "false") \
    .load("dbfs:/FileStore/tables/demo/Out/stations.csv")

bronze_stations.write.format("delta").mode("overwrite").save("/delta/bronze/stations")

# Trips
bronze_trips = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .load("dbfs:/FileStore/tables/demo/Out/trips.csv")

bronze_trips.write.format("delta").mode("overwrite").save("/delta/bronze/trips")

print("Bronze layer criada com sucesso!")
print(f"  riders:   {bronze_riders.count()} registros")
print(f"  payments: {bronze_payments.count()} registros")
print(f"  stations: {bronze_stations.count()} registros")
print(f"  trips:    {bronze_trips.count()} registros")

In [0]:
from pyspark.sql.functions import col, to_date, to_timestamp
from pyspark.sql.types import IntegerType, DoubleType, BooleanType

# ============================================
# CAMADA SILVER - Dados limpos e referenciados
# ============================================

# Silver Riders
silver_riders = spark.read.format("delta").load("/delta/bronze/riders") \
    .select(
        col("_c0").cast(IntegerType()).alias("rider_id"),
        col("_c1").alias("first_name"),
        col("_c2").alias("last_name"),
        col("_c3").alias("address"),
        to_date(col("_c4"), "yyyy-MM-dd").alias("birthday"),
        to_date(col("_c5"), "yyyy-MM-dd").alias("account_start_date"),
        to_date(col("_c6"), "yyyy-MM-dd").alias("account_end_date"),
        (col("_c7") == "True").cast(BooleanType()).alias("is_member")
    )

silver_riders.write.format("delta").mode("overwrite").save("/delta/silver/riders")

# Silver Payments
silver_payments = spark.read.format("delta").load("/delta/bronze/payments") \
    .select(
        col("_c0").cast(IntegerType()).alias("payment_id"),
        to_date(col("_c1"), "yyyy-MM-dd").alias("payment_date"),
        col("_c2").cast(DoubleType()).alias("amount"),
        col("_c3").cast(IntegerType()).alias("rider_id")
    )

silver_payments.write.format("delta").mode("overwrite").save("/delta/silver/payments")

# Silver Stations
silver_stations = spark.read.format("delta").load("/delta/bronze/stations") \
    .select(
        col("_c0").alias("station_id"),
        col("_c1").alias("station_name"),
        col("_c2").cast(DoubleType()).alias("latitude"),
        col("_c3").cast(DoubleType()).alias("longitude")
    )

silver_stations.write.format("delta").mode("overwrite").save("/delta/silver/stations")

# Silver Trips
silver_trips = spark.read.format("delta").load("/delta/bronze/trips") \
    .select(
        col("_c0").alias("trip_id"),
        col("_c1").alias("rideable_type"),
        to_timestamp(col("_c2")).alias("started_at"),
        to_timestamp(col("_c3")).alias("ended_at"),
        col("_c4").alias("start_station_id"),
        col("_c5").alias("end_station_id"),
        col("_c6").cast(IntegerType()).alias("rider_id")
    )

silver_trips.write.format("delta").mode("overwrite").save("/delta/silver/trips")

print("Silver layer criada com sucesso!")
silver_riders.printSchema()
silver_trips.printSchema()

In [0]:
from pyspark.sql.functions import (
    col, year, month, dayofmonth, dayofweek, quarter,
    when, datediff, round as spark_round, lit,
    unix_timestamp, concat_ws, date_format
)

# ============================================
# CAMADA GOLD - Star Schema
# ============================================

# --- DIMENSAO: dim_riders ---
silver_riders = spark.read.format("delta").load("/delta/silver/riders")

dim_riders = silver_riders.select(
    col("rider_id"),
    col("first_name"),
    col("last_name"),
    col("address"),
    col("birthday"),
    col("account_start_date"),
    col("account_end_date"),
    col("is_member")
)

dim_riders.write.format("delta").mode("overwrite").save("/delta/gold/dim_riders")
print(f"dim_riders: {dim_riders.count()} registros")

# --- DIMENSAO: dim_stations ---
silver_stations = spark.read.format("delta").load("/delta/silver/stations")

dim_stations = silver_stations.select(
    col("station_id"),
    col("station_name"),
    col("latitude"),
    col("longitude")
)

dim_stations.write.format("delta").mode("overwrite").save("/delta/gold/dim_stations")
print(f"dim_stations: {dim_stations.count()} registros")

# --- DIMENSAO: dim_date ---
# Gerar tabela de datas a partir do range dos dados
silver_payments = spark.read.format("delta").load("/delta/silver/payments")
silver_trips = spark.read.format("delta").load("/delta/silver/trips")

from pyspark.sql.functions import explode, sequence, to_date as to_date_fn

date_range = spark.sql("""
    SELECT explode(sequence(
        to_date('2013-01-01'), 
        to_date('2022-12-31'), 
        interval 1 day
    )) as date
""")

dim_date = date_range.select(
    date_format(col("date"), "yyyyMMdd").cast(IntegerType()).alias("date_key"),
    col("date"),
    year(col("date")).alias("year"),
    month(col("date")).alias("month"),
    dayofmonth(col("date")).alias("day"),
    dayofweek(col("date")).alias("day_of_week"),
    quarter(col("date")).alias("quarter"),
    when(dayofweek(col("date")).isin(1, 7), True).otherwise(False).alias("is_weekend")
)

dim_date.write.format("delta").mode("overwrite").save("/delta/gold/dim_date")
print(f"dim_date: {dim_date.count()} registros")

In [0]:
from pyspark.sql.functions import (
    col, unix_timestamp, round as spark_round, date_format,
    hour, floor, months_between, year, month
)
from pyspark.sql.types import IntegerType

# ============================================
# TABELAS FATO - Gold Layer (Enriquecidas)
# ============================================

# --- FATO: fact_trips (com campos para Business Outcomes) ---
silver_trips = spark.read.format("delta").load("/delta/silver/trips")
dim_riders = spark.read.format("delta").load("/delta/gold/dim_riders")

fact_trips = silver_trips.join(dim_riders.select("rider_id", "birthday", "is_member"), "rider_id") \
    .select(
        col("trip_id"),
        col("rideable_type"),
        col("started_at"),
        col("ended_at"),
        col("start_station_id"),
        col("end_station_id"),
        col("rider_id"),
        col("is_member"),
        # Duracao em minutos
        spark_round(
            (unix_timestamp(col("ended_at")) - unix_timestamp(col("started_at"))) / 60, 2
        ).alias("duration_minutes"),
        # Hora do dia (para analise por horario)
        hour(col("started_at")).alias("hour_of_day"),
        # Idade do passageiro no momento da viagem
        floor(months_between(col("started_at"), col("birthday")) / 12).cast(IntegerType()).alias("rider_age_at_ride"),
        # Chaves para dimensoes
        date_format(col("started_at"), "yyyyMMdd").cast(IntegerType()).alias("date_key")
    )

fact_trips.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/delta/gold/fact_trips")
print(f"fact_trips: {fact_trips.count()} registros")

# --- FATO: fact_payments (com idade do membro no inicio da conta) ---
silver_payments = spark.read.format("delta").load("/delta/silver/payments")

fact_payments = silver_payments.join(
    dim_riders.select("rider_id", "birthday", "account_start_date", "is_member"), "rider_id"
).select(
    col("payment_id"),
    col("payment_date"),
    col("amount"),
    col("rider_id"),
    col("is_member"),
    # Idade do membro no inicio da conta
    floor(months_between(col("account_start_date"), col("birthday")) / 12).cast(IntegerType()).alias("rider_age_at_account_start"),
    # Chaves para dimensoes
    date_format(col("payment_date"), "yyyyMMdd").cast(IntegerType()).alias("date_key")
)

fact_payments.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/delta/gold/fact_payments")
print(f"fact_payments: {fact_payments.count()} registros")

print("\n=== GOLD LAYER COMPLETA ===")
print("Star Schema criado com sucesso!")
print("  Dimensoes: dim_riders, dim_stations, dim_date")
print("  Fatos:     fact_trips, fact_payments")
print("\nCampos enriquecidos:")
print("  fact_trips:    duration_minutes, hour_of_day, rider_age_at_ride, is_member")
print("  fact_payments: rider_age_at_account_start, is_member")

In [0]:
from pyspark.sql.functions import avg, sum as spark_sum, count, col, when

# ============================================
# ANALISE 1: Quanto tempo e gasto por viagem
# ============================================

fact_trips = spark.read.format("delta").load("/delta/gold/fact_trips")
dim_stations = spark.read.format("delta").load("/delta/gold/dim_stations")
dim_date = spark.read.format("delta").load("/delta/gold/dim_date")

# 1a) Por dia da semana e horario do dia
print("=== DURACAO MEDIA POR DIA DA SEMANA E HORARIO ===")
trips_by_time = fact_trips.join(dim_date, "date_key") \
    .groupBy("day_of_week", "hour_of_day") \
    .agg(
        avg("duration_minutes").alias("avg_duration_min"),
        count("trip_id").alias("total_trips")
    ) \
    .orderBy("day_of_week", "hour_of_day")

display(trips_by_time)

# 1b) Por estacao de partida
print("\n=== DURACAO MEDIA POR ESTACAO DE PARTIDA (TOP 20) ===")
trips_by_start_station = fact_trips \
    .join(dim_stations, fact_trips.start_station_id == dim_stations.station_id) \
    .groupBy("station_name") \
    .agg(
        avg("duration_minutes").alias("avg_duration_min"),
        count("trip_id").alias("total_trips")
    ) \
    .orderBy(col("total_trips").desc()) \
    .limit(20)

display(trips_by_start_station)

# 1c) Por idade do passageiro no momento da viagem
print("\n=== DURACAO MEDIA POR FAIXA ETARIA ===")
trips_by_age = fact_trips \
    .withColumn("age_group", 
        when(col("rider_age_at_ride") < 25, "<25")
        .when(col("rider_age_at_ride") < 35, "25-34")
        .when(col("rider_age_at_ride") < 45, "35-44")
        .when(col("rider_age_at_ride") < 55, "45-54")
        .otherwise("55+")
    ) \
    .groupBy("age_group") \
    .agg(
        avg("duration_minutes").alias("avg_duration_min"),
        count("trip_id").alias("total_trips")
    ) \
    .orderBy("age_group")

display(trips_by_age)

# 1d) Por membro vs passageiro casual
print("\n=== DURACAO MEDIA: MEMBRO vs CASUAL ===")
trips_by_member = fact_trips \
    .groupBy("is_member") \
    .agg(
        avg("duration_minutes").alias("avg_duration_min"),
        count("trip_id").alias("total_trips"),
        spark_sum("duration_minutes").alias("total_minutes")
    )

display(trips_by_member)

In [0]:
from pyspark.sql.functions import avg, sum as spark_sum, count, col, when, floor, months_between
from pyspark.sql.types import IntegerType

# ============================================
# ANALISE 2: Quanto dinheiro e gasto
# ============================================

fact_payments = spark.read.format("delta").load("/delta/gold/fact_payments")
dim_date = spark.read.format("delta").load("/delta/gold/dim_date")

# 2a) Por mes, trimestre e ano
print("=== RECEITA POR MES/TRIMESTRE/ANO ===")
revenue_by_period = fact_payments.join(dim_date, "date_key") \
    .groupBy("year", "quarter", "month") \
    .agg(
        spark_sum("amount").alias("total_revenue"),
        count("payment_id").alias("num_payments"),
        avg("amount").alias("avg_payment")
    ) \
    .orderBy("year", "month")

display(revenue_by_period)

# 2b) Por membro, com base na idade do usuario no inicio da conta
print("\n=== GASTO POR MEMBRO E FAIXA ETARIA (idade no inicio da conta) ===")
revenue_by_member_age = fact_payments \
    .withColumn("age_group_at_signup",
        when(col("rider_age_at_account_start") < 25, "<25")
        .when(col("rider_age_at_account_start") < 35, "25-34")
        .when(col("rider_age_at_account_start") < 45, "35-44")
        .when(col("rider_age_at_account_start") < 55, "45-54")
        .otherwise("55+")
    ) \
    .groupBy("is_member", "age_group_at_signup") \
    .agg(
        spark_sum("amount").alias("total_spent"),
        count("payment_id").alias("num_payments"),
        avg("amount").alias("avg_payment")
    ) \
    .orderBy("is_member", "age_group_at_signup")

display(revenue_by_member_age)

In [0]:
from pyspark.sql.functions import (
    avg, sum as spark_sum, count, col, countDistinct,
    year, month, concat_ws, round as spark_round
)

# ============================================
# EXTRA CREDIT: Gasto por membro
# ============================================

fact_trips = spark.read.format("delta").load("/delta/gold/fact_trips")
fact_payments = spark.read.format("delta").load("/delta/gold/fact_payments")
dim_date = spark.read.format("delta").load("/delta/gold/dim_date")

# Calcular metricas mensais por rider
# a) Media de viagens por mes por membro
trips_monthly = fact_trips.join(dim_date, "date_key") \
    .groupBy("rider_id", "is_member", "year", "month") \
    .agg(
        count("trip_id").alias("trips_in_month"),
        spark_sum("duration_minutes").alias("minutes_in_month")
    )

rider_monthly_avg = trips_monthly.groupBy("rider_id", "is_member") \
    .agg(
        spark_round(avg("trips_in_month"), 2).alias("avg_trips_per_month"),
        spark_round(avg("minutes_in_month"), 2).alias("avg_minutes_per_month")
    )

# Calcular gasto total por membro
rider_spending = fact_payments.groupBy("rider_id") \
    .agg(
        spark_sum("amount").alias("total_spent")
    )

# Juntar tudo: gasto + media de viagens/mes + media de minutos/mes
print("=== GASTO POR MEMBRO: MEDIA DE VIAGENS/MES ===")
extra_credit = rider_spending \
    .join(rider_monthly_avg, "rider_id") \
    .select(
        col("rider_id"),
        col("is_member"),
        col("total_spent"),
        col("avg_trips_per_month"),
        col("avg_minutes_per_month")
    ) \
    .orderBy(col("total_spent").desc())

display(extra_credit)

# Resumo agregado: gasto medio por faixa de frequencia
print("\n=== GASTO MEDIO POR FAIXA DE FREQUENCIA (viagens/mes) ===")
from pyspark.sql.functions import when

spending_by_frequency = extra_credit \
    .withColumn("frequency_group",
        when(col("avg_trips_per_month") < 5, "1-4 viagens/mes")
        .when(col("avg_trips_per_month") < 10, "5-9 viagens/mes")
        .when(col("avg_trips_per_month") < 20, "10-19 viagens/mes")
        .otherwise("20+ viagens/mes")
    ) \
    .groupBy("is_member", "frequency_group") \
    .agg(
        avg("total_spent").alias("avg_total_spent"),
        avg("avg_minutes_per_month").alias("avg_min_per_month"),
        count("rider_id").alias("num_riders")
    ) \
    .orderBy("is_member", "frequency_group")

display(spending_by_frequency)